# G14 — GT-Guided Prototype Counterfactual

**Motivation:** The cross-model heatmap transplant in G13 was confounded by separately
trained encoders: swapping heatmaps across models mixes incompatible feature distributions,
making the Δ-Dice uninterpretable.

**Correct approach:** Stay within one model's feature space. For the skip models, replace
the learned prototype heatmaps with *GT-guided centroid heatmaps* computed from the
same model's encoder features and the ground-truth segmentation mask.

**GT-centroid heatmap construction (per level l, per class k):**
1. Run skip model encoder → `F_l` shape `(1, C_l, H_l, W_l)`
2. Downsample GT to `(H_l, W_l)` via nearest-neighbor
3. `p_k = mean(F_l[:, :, GT_l==k], dim=-1)` — class centroid in feature space
4. `H_k = exp(-||F_l - p_k||² / σ²)` — Gaussian heatmap, shape `(H_l, W_l)`
5. Stack → `{l: (1, K, 1, H_l, W_l)}` to match SoftMask input format

**What we measure:**
- Dice(logits_GT_guided) vs Dice(logits_learned_proto)
- Heatmap AP of GT-guided heatmaps vs learned prototype heatmaps

**Models tested:**
- skip-16 (`proto_seg_ct_l4only.pth`, levels=[4])
- skip-32+16 (`proto_seg_ct_l3l4_warmstart.pth`, levels=[3,4])

**Success criteria:**
- GT-guided AP >> learned-proto AP → GT heatmaps are better localized
- If Dice(GT) ≈ Dice(orig): decoder doesn't use heatmap location → bypass active
- If Dice(GT) >> Dice(orig): better heatmaps → better segmentation (interpretability wins)

In [1]:
import sys, os
_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(_root)
sys.path.insert(0, _root)
os.environ.setdefault('PYTORCH_MPS_HIGH_WATERMARK_RATIO', '0.0')
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from pathlib import Path

from src.data.mmwhs_dataset import MMWHSSliceDataset, NUM_CLASSES, LABEL_NAMES
from src.models.proto_seg_net import ProtoSegNet

DEVICE   = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
DATA_DIR = 'data/pack/processed_data'
CKPT_DIR = Path('checkpoints')
OUT_DIR  = Path('results/v11')
OUT_DIR.mkdir(parents=True, exist_ok=True)
MAX_SLICES = 100   # slices per experiment (test set: 484 total)
SIGMA = 1.0        # bandwidth for Gaussian heatmap (in feature-space L2 units)

print(f'Device: {DEVICE}')

Device: mps


In [2]:
# Load test images and labels
test_ds = MMWHSSliceDataset(DATA_DIR, 'ct', 'test', augment=False, preload=True)
imgs_test   = torch.stack([test_ds[i]['image'] for i in range(len(test_ds))])
labels_test = torch.stack([test_ds[i]['label'] for i in range(len(test_ds))])
print(f'Test images: {tuple(imgs_test.shape)},  labels: {tuple(labels_test.shape)}')

Test images: (484, 1, 256, 256),  labels: (484, 256, 256)


In [3]:
# ── Model loading ─────────────────────────────────────────────────────────────

def load_skip_model(ckpt_name: str, proto_levels: list[int]) -> ProtoSegNet:
    ckpt = torch.load(CKPT_DIR / ckpt_name, map_location='cpu', weights_only=False)
    model = ProtoSegNet(
        n_classes=NUM_CLASSES,
        proto_levels=proto_levels,
        use_level_attention=ckpt.get('use_level_attention', False),
    )
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval().to(DEVICE)
    return model

print('Loading skip-16  (L4)…')
m_skip16   = load_skip_model('proto_seg_ct_l4only.pth', [4])
print('Loading skip-32+16 (L3+L4)…')
m_skip3416 = load_skip_model('proto_seg_ct_l3l4_warmstart.pth', [3, 4])
print('Done.')

Loading skip-16  (L4)…
Loading skip-32+16 (L3+L4)…
Done.


In [4]:
# ── GT-guided heatmap construction ────────────────────────────────────────────

def make_gt_heatmaps(
    feat: dict[int, torch.Tensor],   # {l: (1, C_l, H_l, W_l)}
    gt: torch.Tensor,                # (1, H, W) long, GT label at full resolution
    proto_levels: list[int],
    n_classes: int = NUM_CLASSES,
    sigma: float = SIGMA,
) -> dict[int, torch.Tensor]:        # {l: (1, K, 1, H_l, W_l)}
    """
    Build GT-centroid heatmaps inside the model's own feature space.

    For each level l and class k:
      1. Downsample GT to (H_l, W_l) via nearest-neighbor
      2. Compute class centroid p_k = mean of F_l at GT==k pixels
         (if class absent in this slice, p_k = zero vector)
      3. H_k(i,j) = exp(-||F_l(i,j) - p_k||^2 / sigma^2)
    """
    heatmaps_gt = {}
    for l in proto_levels:
        F_l = feat[l]               # (1, C_l, H_l, W_l)
        H_l, W_l = F_l.shape[-2], F_l.shape[-1]
        C_l = F_l.shape[1]

        # Downsample GT
        gt_l = F.interpolate(
            gt.float().unsqueeze(0),  # (1, 1, H, W)
            size=(H_l, W_l),
            mode='nearest',
        ).long().squeeze(0).squeeze(0)  # (H_l, W_l)

        # Feature map flattened: (H_l*W_l, C_l)
        F_flat = F_l[0].view(C_l, -1).T.float()  # (H_l*W_l, C_l)

        H_all = []
        for k in range(n_classes):
            mask_k = (gt_l == k).view(-1)   # (H_l*W_l,)
            if mask_k.sum() > 0:
                p_k = F_flat[mask_k].mean(dim=0)   # (C_l,)
            else:
                p_k = torch.zeros(C_l, device=F_l.device, dtype=F_flat.dtype)

            dist_sq = ((F_flat - p_k.unsqueeze(0)) ** 2).sum(dim=1)   # (H_l*W_l,)
            H_k = torch.exp(-dist_sq / (sigma ** 2))                   # (H_l*W_l,)
            H_all.append(H_k.view(H_l, W_l))

        # Stack → (K, H_l, W_l) → (1, K, 1, H_l, W_l)
        H_stacked = torch.stack(H_all, dim=0).unsqueeze(0).unsqueeze(2)  # (1, K, 1, H_l, W_l)
        heatmaps_gt[l] = H_stacked

    return heatmaps_gt

In [5]:
# ── Partial forward: encode → GT-heatmap → soft-mask → decode ────────────────

@torch.no_grad()
def forward_with_custom_heatmaps(
    model: ProtoSegNet,
    x: torch.Tensor,                       # (1, 1, H, W)
    heatmaps_custom: dict[int, torch.Tensor],  # {l: (1, K, 1, H_l, W_l)}
) -> torch.Tensor:                          # logits (1, K, H, W)
    """
    Run the model's encode → soft-mask → decode path,
    substituting custom heatmaps for the prototype heatmaps.
    All other decoder operations (skip connections, etc.) are identical to normal.
    """
    feat = model.encoder(x)  # {1: (1,32,128,128), 2: (1,64,64,64), 3: (1,128,32,32), 4: (1,256,16,16)}

    masked = {}
    for l in [1, 2, 3, 4]:
        if str(l) in model.proto_layers:
            if l in heatmaps_custom:
                A = heatmaps_custom[l]   # (1, K, 1, H_l, W_l)
                masked[l] = model.mask_module(A, feat[l])
            else:
                # Level not in custom heatmaps → fall back to learned prototypes
                A = model._proto_layer(l)(feat[l])
                masked[l] = model.mask_module(A, feat[l])
        else:
            masked[l] = feat[l]

    d = model.dec4(masked[4], masked[3])
    d = model.dec3(d, masked[2])
    d = model.dec2(d, masked[1])
    d = F.interpolate(d, scale_factor=2, mode='bilinear', align_corners=False)
    d = model.dec1(d)
    return model.final_conv(d)   # (1, K, H, W)

In [6]:
# ── Dice computation ──────────────────────────────────────────────────────────

def dice_per_class(
    logits: torch.Tensor,   # (1, K, H, W)
    gt: torch.Tensor,       # (1, H, W)  long
    n_classes: int = NUM_CLASSES,
    smooth: float = 1e-6,
) -> np.ndarray:             # (K,)
    pred = logits[0].argmax(dim=0)   # (H, W)
    gt0  = gt[0]                     # (H, W)
    dices = []
    for k in range(n_classes):
        p = (pred == k).float()
        g = (gt0  == k).float()
        inter = (p * g).sum()
        dices.append(((2 * inter + smooth) / (p.sum() + g.sum() + smooth)).item())
    return np.array(dices)

In [7]:
# ── Heatmap AP utility ─────────────────────────────────────────────────────────

from sklearn.metrics import average_precision_score

def heatmap_ap_single(
    heatmaps: dict[int, torch.Tensor],   # {l: (1, K, M, H_l, W_l)}
    gt: torch.Tensor,                    # (1, H, W) long
    n_classes: int = NUM_CLASSES,
) -> float:
    """
    Aggregate AP across all levels and classes (foreground only).
    Upsamples each heatmap to full resolution, then computes per-class AP.
    """
    H, W = gt.shape[-2], gt.shape[-1]
    aps = []
    for l, A in heatmaps.items():
        # A: (1, K, M, H_l, W_l) — max over M
        A_max = A[0].max(dim=1).values   # (K, H_l, W_l)
        A_up  = F.interpolate(
            A_max.unsqueeze(0), size=(H, W), mode='bilinear', align_corners=False
        )[0]   # (K, H, W)
        gt0 = gt[0].cpu().numpy().ravel()  # (H*W,)
        for k in range(1, n_classes):      # skip background
            score = A_up[k].cpu().numpy().ravel()
            label = (gt0 == k).astype(np.float32)
            if label.sum() == 0:
                continue
            aps.append(average_precision_score(label, score))
    return float(np.mean(aps)) if aps else float('nan')

In [8]:
# ── Main experiment loop ───────────────────────────────────────────────────────

def run_counterfactual_experiment(
    model: ProtoSegNet,
    imgs: torch.Tensor,    # (S, 1, H, W)
    labels: torch.Tensor,  # (S, H, W) long
    model_name: str,
    max_slices: int = MAX_SLICES,
    sigma: float = SIGMA,
) -> pd.DataFrame:
    model.eval()
    S = imgs.shape[0]
    indices = torch.linspace(0, S - 1, min(max_slices, S)).long().tolist()

    rows = []
    for i in indices:
        x  = imgs[i:i+1].to(DEVICE)     # (1, 1, H, W)
        gt = labels[i:i+1].to(DEVICE)   # (1, H, W)

        with torch.no_grad():
            # ── Original forward ──────────────────────────────────────────────
            logits_orig, hm_learned = model(x)
            dice_orig = dice_per_class(logits_orig, gt)
            ap_learned = heatmap_ap_single(hm_learned, gt)

            # ── GT-guided forward ─────────────────────────────────────────────
            feat = model.encoder(x)
            hm_gt = make_gt_heatmaps(feat, gt, model.proto_levels, sigma=sigma)
            logits_gt = forward_with_custom_heatmaps(model, x, hm_gt)
            dice_gt = dice_per_class(logits_gt, gt)
            ap_gt   = heatmap_ap_single(hm_gt, gt)

        # Mean Dice over foreground (classes 1–7)
        rows.append({
            'slice': i,
            'model': model_name,
            'dice_orig_mean': dice_orig[1:].mean(),
            'dice_gt_mean':   dice_gt[1:].mean(),
            'delta_dice':     dice_gt[1:].mean() - dice_orig[1:].mean(),
            'ap_learned':     ap_learned,
            'ap_gt':          ap_gt,
            'delta_ap':       ap_gt - ap_learned,
        })

    return pd.DataFrame(rows)

print('Setup complete — ready to run experiments.')

Setup complete — ready to run experiments.


In [9]:
# ── Experiment 1: skip-16 (L4 only) ───────────────────────────────────────────
print('Running skip-16 (L4) counterfactual…')
df_skip16 = run_counterfactual_experiment(
    m_skip16, imgs_test, labels_test, model_name='skip-16 (L4)'
)
df_skip16.to_csv(OUT_DIR / 'counterfactual_gt_guided_skip16.csv', index=False)

print(f"  Dice orig:    {df_skip16['dice_orig_mean'].mean():.4f}")
print(f"  Dice GT:      {df_skip16['dice_gt_mean'].mean():.4f}")
print(f"  Δ Dice:       {df_skip16['delta_dice'].mean():+.4f}  (std {df_skip16['delta_dice'].std():.4f})")
print(f"  AP learned:   {df_skip16['ap_learned'].mean():.4f}")
print(f"  AP GT-guided: {df_skip16['ap_gt'].mean():.4f}")
print(f"  Δ AP:         {df_skip16['delta_ap'].mean():+.4f}")
del m_skip16

Running skip-16 (L4) counterfactual…


  Dice orig:    0.8226
  Dice GT:      0.2342
  Δ Dice:       -0.5884  (std 0.1762)
  AP learned:   0.5512
  AP GT-guided: 0.6228
  Δ AP:         +0.0716


In [10]:
# ── Experiment 2: skip-32+16 (L3+L4) ─────────────────────────────────────────
print('Running skip-32+16 (L3+L4) counterfactual…')
df_skip3416 = run_counterfactual_experiment(
    m_skip3416, imgs_test, labels_test, model_name='skip-32+16 (L3+L4)'
)
df_skip3416.to_csv(OUT_DIR / 'counterfactual_gt_guided_skip3416.csv', index=False)

print(f"  Dice orig:    {df_skip3416['dice_orig_mean'].mean():.4f}")
print(f"  Dice GT:      {df_skip3416['dice_gt_mean'].mean():.4f}")
print(f"  Δ Dice:       {df_skip3416['delta_dice'].mean():+.4f}  (std {df_skip3416['delta_dice'].std():.4f})")
print(f"  AP learned:   {df_skip3416['ap_learned'].mean():.4f}")
print(f"  AP GT-guided: {df_skip3416['ap_gt'].mean():.4f}")
print(f"  Δ AP:         {df_skip3416['delta_ap'].mean():+.4f}")
del m_skip3416

Running skip-32+16 (L3+L4) counterfactual…


  Dice orig:    0.8123
  Dice GT:      0.1964
  Δ Dice:       -0.6159  (std 0.1885)
  AP learned:   0.0775
  AP GT-guided: 0.3890
  Δ AP:         +0.3115


In [11]:
# ── Summary table ──────────────────────────────────────────────────────────────
df_all = pd.concat([df_skip16, df_skip3416], ignore_index=True)
df_all.to_csv(OUT_DIR / 'counterfactual_gt_guided_all.csv', index=False)

summary = df_all.groupby('model').agg(
    dice_orig   =('dice_orig_mean', 'mean'),
    dice_gt     =('dice_gt_mean',   'mean'),
    delta_dice  =('delta_dice',     'mean'),
    ap_learned  =('ap_learned',     'mean'),
    ap_gt       =('ap_gt',          'mean'),
    delta_ap    =('delta_ap',       'mean'),
).reset_index()

print('='*80)
print('  GT-Guided Counterfactual: Summary')
print('='*80)
print(summary.round(4).to_string(index=False))
print()
print('Interpretation:')
print('  AP Δ > 0      → GT heatmaps better localized than learned prototypes')
print('  Dice Δ ≈ 0    → bypass active: better heatmaps do NOT improve segmentation')
print('  Dice Δ >> 0   → no bypass: decoder uses heatmap location (interpretable)')

  GT-Guided Counterfactual: Summary
             model  dice_orig  dice_gt  delta_dice  ap_learned  ap_gt  delta_ap
      skip-16 (L4)     0.8226   0.2342     -0.5884      0.5512 0.6228    0.0716
skip-32+16 (L3+L4)     0.8123   0.1964     -0.6159      0.0775 0.3890    0.3115

Interpretation:
  AP Δ > 0      → GT heatmaps better localized than learned prototypes
  Dice Δ ≈ 0    → bypass active: better heatmaps do NOT improve segmentation
  Dice Δ >> 0   → no bypass: decoder uses heatmap location (interpretable)


In [12]:
# ── Visualise Δ Dice distributions ────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, df) in zip(axes, [('skip-16 (L4)', df_skip16), ('skip-32+16 (L3+L4)', df_skip3416)]):
    ax.hist(df['delta_dice'], bins=20, color='steelblue', edgecolor='white', alpha=0.8)
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
    ax.axvline(df['delta_dice'].mean(), color='orange', linestyle='-', linewidth=2,
               label=f'mean={df["delta_dice"].mean():+.4f}')
    ax.set_title(f'{name}\nΔ Dice (GT − orig)')
    ax.set_xlabel('Δ mean Dice (fg)')
    ax.legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'counterfactual_gt_guided_delta_dice.png', dpi=150)
plt.close()
print('Saved: results/v11/counterfactual_gt_guided_delta_dice.png')

Saved: results/v11/counterfactual_gt_guided_delta_dice.png


In [13]:
# ── Sigma sensitivity check ────────────────────────────────────────────────────
# Run skip-16 with sigma=0.5, 2.0, 5.0 to verify robustness

print('Reloading skip-16 for sigma sweep…')
m_skip16_sigma = load_skip_model('proto_seg_ct_l4only.pth', [4])

sigma_rows = []
for sigma_val in [0.5, 1.0, 2.0, 5.0]:
    df_s = run_counterfactual_experiment(
        m_skip16_sigma, imgs_test, labels_test,
        model_name='skip-16', max_slices=50, sigma=sigma_val
    )
    sigma_rows.append({
        'sigma': sigma_val,
        'delta_dice': df_s['delta_dice'].mean(),
        'ap_gt':      df_s['ap_gt'].mean(),
    })
    print(f'  sigma={sigma_val:.1f}: Δ Dice={df_s["delta_dice"].mean():+.4f}, AP_GT={df_s["ap_gt"].mean():.4f}')

del m_skip16_sigma
print('\nSigma sensitivity summary:')
print(pd.DataFrame(sigma_rows).round(4).to_string(index=False))

Reloading skip-16 for sigma sweep…


  sigma=0.5: Δ Dice=-0.5930, AP_GT=0.5743


  sigma=1.0: Δ Dice=-0.5912, AP_GT=0.6215


  sigma=2.0: Δ Dice=-0.5968, AP_GT=0.6677


  sigma=5.0: Δ Dice=-0.4378, AP_GT=0.6821

Sigma sensitivity summary:
 sigma  delta_dice  ap_gt
   0.5     -0.5930 0.5743
   1.0     -0.5912 0.6215
   2.0     -0.5968 0.6677
   5.0     -0.4378 0.6821
